# SFR Training

This notebook contains a standalone version of the saliency-feedback regularization (SFR) training workflow used in the main 3D CNN notebook.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, CSVLogger, EarlyStopping
from tensorflow.keras.utils import to_categorical

from utils.processamento_dados import load_nifti_data_balanced_preallocated, nifti_data_generator_3d
from utils.metricas_e_visualizacao import plot_training_history, get_predictions, get_classification_report, plot_confusion_matrix

print("TensorFlow version:", tf.__version__)

In [ ]:
# Configuração do experimento
base_dir = r"/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_NORMALIZED"
class_names = ['cn', 'ad']
input_shape = (80, 96, 80, 1)

train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
results_dir = os.path.join(base_dir, 'results', 'sfr_training')
os.makedirs(results_dir, exist_ok=True)

print(f"Training data: {train_dir}")
print(f"Validation data: {val_dir}")
print(f"Results dir: {results_dir}")

In [ ]:
# Carregar dados balanceados
train_images, train_labels, train_paths, _ = load_nifti_data_balanced_preallocated(
    train_dir,
    class_names,
    augment=False,
    target_per_class=1000,
)

val_images, val_labels, val_paths, _ = load_nifti_data_balanced_preallocated(
    val_dir,
    class_names,
    augment=False,
    target_per_class=500,
)

print("Train shape:", train_images.shape)
print("Validation shape:", val_images.shape)

In [ ]:
# Helper functions for the SFR loss

def create_model_3d(input_shape, n_classes):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Conv3D(8, (3, 3, 3), activation='relu', padding='same')(inputs)
    x = tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2))(x)
    x = tf.keras.layers.Conv3D(16, (3, 3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2))(x)
    x = tf.keras.layers.Conv3D(32, (3, 3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)


def load_mask(mask_path, target_shape):
    mask = tf.keras.preprocessing.image.load_img(mask_path, color_mode='grayscale')
    mask_array = np.array(mask)
    mask_array = mask_array.astype(np.float32)
    mask_array = zoom(mask_array, (target_shape[0] / mask_array.shape[0], target_shape[1] / mask_array.shape[1]), order=0)
    mask_array = (mask_array > 0.5).astype(np.float32)
    mask_array = 1.0 - mask_array
    return tf.cast(mask_array, tf.float32)

In [ ]:
# Load a simple mask for the SFR penalty
mask_path = os.path.join(base_dir, 'mask.png')
if os.path.exists(mask_path):
    mask_crop = load_mask(mask_path, (80, 96))
else:
    mask_crop = tf.zeros((80, 96), dtype=tf.float32)

print("Mask shape:", mask_crop.shape)

In [ ]:
class SFRModel(tf.keras.Model):
    def __init__(self, model, alpha, mask_crop, last_conv_layer_name, **kwargs):
        super().__init__(**kwargs)
        self.model = model
        self.alpha = alpha
        self.mask_crop = tf.cast(mask_crop, tf.float32)
        self.last_conv_layer_name = last_conv_layer_name
        self.grad_model = tf.keras.Model(
            [self.model.inputs],
            [self.model.get_layer(last_conv_layer_name).output, self.model.output]
        )
        self.saliency_tracker = tf.keras.metrics.Mean(name="saliency_loss")
        self.classification_tracker = tf.keras.metrics.Mean(name="classification_loss")

    def train_step(self, data):
        x, y = data
        x = tf.cast(x, tf.float32)
        y = tf.cast(y, tf.float32)

        with tf.GradientTape() as total_tape:
            with tf.GradientTape() as gradcam_tape:
                conv_outputs, predictions = self.grad_model(x, training=True)
                class_loss = predictions[:, 1]

            grads = gradcam_tape.gradient(class_loss, conv_outputs)
            pooled_grads = tf.reduce_mean(grads, axis=(1, 2, 3))
            heatmap = tf.reduce_sum(
                tf.multiply(pooled_grads[:, tf.newaxis, tf.newaxis, tf.newaxis, :], conv_outputs),
                axis=-1,
            )
            heatmap = tf.nn.relu(heatmap)
            heatmap_norm = heatmap / (tf.reduce_sum(heatmap, axis=(1, 2, 3), keepdims=True) + 1e-6)
            saliency_penalty_batch = tf.reduce_sum(heatmap_norm * self.mask_crop, axis=(1, 2, 3))
            mean_saliency_penalty = tf.reduce_mean(saliency_penalty_batch)

            main_loss = self.compiled_loss(y, predictions, regularization_losses=self.losses)
            total_loss = main_loss + (self.alpha * mean_saliency_penalty)

        gradients = total_tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        self.saliency_tracker.update_state(mean_saliency_penalty)
        self.classification_tracker.update_state(main_loss)
        self.compiled_metrics.update_state(y, predictions)

        return {m.name: m.result() for m in self.metrics}

    def call(self, x):
        return self.model(x)

    @property
    def metrics(self):
        return [self.saliency_tracker, self.classification_tracker] + super().metrics

In [ ]:
# Build the model
base_model = create_model_3d(input_shape, len(class_names))
sfr_model = SFRModel(model=base_model, alpha=0.01, mask_crop=mask_crop, last_conv_layer_name='conv3d_2')
sfr_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)
sfr_model.build(input_shape)

sfr_model.summary()

In [ ]:
# Training generators
batch_size = 8
steps_per_epoch = len(train_images) // batch_size
validation_steps = len(val_images) // batch_size

train_generator = nifti_data_generator_3d(train_images, train_labels, batch_size)
val_generator = nifti_data_generator_3d(val_images, val_labels, batch_size)

epochs = 20

new_model_name_ker = f"sfr_model_{epochs}_epochs_batch_{batch_size}_{len(class_names)}_classes.keras"

callbacks = [
    ModelCheckpoint(filepath=os.path.join(results_dir, new_model_name_ker), monitor='val_categorical_accuracy', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1),
    CSVLogger(os.path.join(results_dir, 'training_log.csv'), append=False),
    EarlyStopping(monitor='val_loss', patience=10, verbose=1)
]

In [ ]:
print("Iniciando treinamento SFR")
history = sfr_model.fit(
    train_generator,
    epochs=epochs,
    verbose=1,
    validation_data=val_generator,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
)

In [ ]:
plot_training_history(history, results_dir)

In [ ]:
# Optional evaluation
pred_labels, true_labels, pred = get_predictions(val_images, val_labels, batch_size, sfr_model)
get_classification_report(true_labels, pred_labels, results_dir, 'val')
plot_confusion_matrix(true_labels, pred_labels, results_dir, 'val', class_names)